# Module 1.7 — Capstone: Build a RAG System and Evaluate It End-to-End

Every prior notebook in Module 1 either used hand-constructed test cases (1.2–1.4) or a fixture-data toy graph (1.5). This capstone does the real thing: build an actual RAG system over a real (small) document set, generate a **synthetic golden reference dataset** instead of hand-writing test cases one at a time, run the pipeline against every golden question, and score the results with the full metric suite from across Module 1 — in one pass.

_Source: merged from `RAG_Evaluation/Build_RAG_Pipeline_with_Source.ipynb` and `RAG_Evaluation/4. End_to_End_RAG_System_Evaluation.ipynb` (the second notebook re-derives the first's build steps almost verbatim before adding the evaluation workflow; merged here into one build → evaluate flow). **API note:** both originals call `deepeval.synthesizer.Synthesizer` with a `generate_goldens(...)` method and an `embedder=` argument — that API has changed in the `deepeval==4.1.8` this tutorial targets (see the note before the synthesizer cell below); this notebook uses the current method name and config-object shape.

## Part A — Build the RAG system

In [ ]:
# ============ SETUP ============
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY (shell env or .env) before running this notebook."

from langchain_openai import OpenAIEmbeddings

openai_embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
print("Embedding model ready")

### Load and process the dataset

A small (10-document) CSV of short reference articles (`id`, `title`, `context`), living at `07_Advanced_Agentic_Systems/data/rag_eval_docs.csv` — shared across this phase, reached the same way `RAG_Evaluation/`'s own notebooks reach it (`../../data/...`), since this tutorial folder sits at the same depth.

In [ ]:
# ============ LOAD & PROCESS DOCUMENTS ============
import pandas as pd
from langchain_core.documents import Document

df = pd.read_csv("../../data/rag_eval_docs.csv")
print(f"{len(df)} documents loaded")

docs = df.to_dict(orient="records")
processed_docs = [
    Document(page_content=doc["context"], metadata={"title": doc["title"], "id": doc["id"]})
    for doc in docs
]
processed_docs[:2]

### Index into a Chroma vector store

In [ ]:
# ============ INDEX INTO CHROMA ============
from langchain_chroma import Chroma

# A persist_directory scoped to this capstone notebook -- deliberately not reusing
# RAG_Evaluation/my_db/, so this capstone's index stays isolated from that notebook's.
chroma_db = Chroma.from_documents(
    documents=processed_docs,
    collection_name="capstone_db",
    embedding=openai_embed_model,
    collection_metadata={"hnsw:space": "cosine"},  # cosine, not the euclidean default
    persist_directory="./capstone_rag_db",
)

similarity_retriever = chroma_db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.3},
)
print("Retriever ready")

In [ ]:
# ============ SANITY-CHECK RETRIEVAL ============
for query in ["what is AI?", "how do plants survive?"]:
    print(f"Query: {query}")
    for doc in similarity_retriever.invoke(query):
        print(f"  - [{doc.metadata['title']}] {doc.page_content[:80]}...")
    print()

### Build the generation chain

In [ ]:
# ============ BUILD THE RAG CHAIN ============
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

rag_prompt = ChatPromptTemplate.from_template('''You are an assistant who is an expert in question-answering tasks.
Answer the following question using only the following pieces of retrieved context.
If the answer is not in the context, do not make up answers, just say that you don't know.
Keep the answer to the point based on the information from the context.

Question:
{question}

Context:
{context}

Answer:
''')

chatgpt = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


response_chain = (
    {
        "context": itemgetter("context") | RunnableLambda(format_docs),
        "question": itemgetter("question"),
    }
    | rag_prompt
    | chatgpt
    | StrOutputParser()
)

# Keeps the retrieved Documents alongside the generated response, so downstream
# evaluation can build retrieval_context from the SAME chunks the answer was generated from.
rag_chain_w_sources = (
    {"context": similarity_retriever, "question": RunnablePassthrough()}
    | RunnablePassthrough.assign(response=response_chain)
)

result = rag_chain_w_sources.invoke("What is AI?")
print("Answer:", result["response"])
print("Retrieved from:", [d.metadata["title"] for d in result["context"]])

## Part B — Generate a synthetic golden reference dataset

Every prior notebook's test cases were hand-written. That doesn't scale past a handful of examples — DeepEval's `Synthesizer` generates question/expected-answer pairs directly from your own documents instead, so you get a labeled eval set without writing reference answers by hand.

**API note (deepeval `4.1.8`):** the source notebooks call `Synthesizer(model=..., embedder=...)` then `.generate_goldens(contexts=..., evolutions={...}, num_evolutions=..., scenario=..., task=...)`. In the version this tutorial targets, the `embedder` argument is gone, the method is `generate_goldens_from_contexts(...)`, and `evolutions`/`num_evolutions` move into an `EvolutionConfig` object (`scenario`/`task` into a `StylingConfig`) passed to the `Synthesizer` constructor instead. Same underlying idea — evolve each generated question along configurable dimensions (require multi-step reasoning, pull from multiple context pieces, etc.) — just reorganized into config objects.

In [ ]:
# ============ SYNTHESIZE GOLDEN QUESTIONS FROM YOUR OWN DOCS ============
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import EvolutionConfig, StylingConfig
from deepeval.synthesizer.types import Evolution

doc_contexts = [doc.page_content for doc in processed_docs]

synthesizer = Synthesizer(
    model="gpt-4o",
    evolution_config=EvolutionConfig(
        num_evolutions=1,
        evolutions={
            Evolution.REASONING: 0.1,      # evolves the input to require multi-step logical thinking
            Evolution.MULTICONTEXT: 0.9,   # ensures relevant information from the context is required
        },
    ),
    styling_config=StylingConfig(scenario="Retrieval Augmented Generation", task="Question Answering"),
)

goldens = synthesizer.generate_goldens_from_contexts(
    contexts=[[doc] for doc in doc_contexts],  # one context group per document
    include_expected_output=True,
    max_goldens_per_context=1,
)

print(f"Generated {len(goldens)} golden questions")
print()
print("Input:          ", goldens[0].input)
print("Expected output: ", goldens[0].expected_output)

### Run each golden question through the RAG pipeline, building test cases

In [ ]:
# ============ GOLDENS -> TEST CASES (run the real pipeline on each) ============
from typing import List

from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
from tqdm import tqdm


def convert_goldens_to_test_cases(goldens: List[Golden]) -> List[LLMTestCase]:
    test_cases = []
    for golden in tqdm(goldens):
        response_obj = rag_chain_w_sources.invoke(golden.input)
        test_cases.append(LLMTestCase(
            input=golden.input,
            actual_output=response_obj["response"],
            expected_output=golden.expected_output,
            context=golden.context,
            retrieval_context=[doc.page_content for doc in response_obj["context"]],
        ))
    return test_cases


eval_dataset = EvaluationDataset()
eval_dataset.goldens = goldens
eval_dataset.test_cases = convert_goldens_to_test_cases(eval_dataset.goldens)

print(f"Built {len(eval_dataset.test_cases)} test cases from live pipeline runs")

**What just happened, and why it matters:** every test case's `retrieval_context` comes from *actually running the retriever* on `golden.input`, not from the documents the golden was generated from. If the retriever performs badly on a given question, its `retrieval_context` will reflect that — the evaluation below is testing the real, currently-running pipeline end to end, not just checking whether the generator can produce a good answer given hand-picked perfect context (which is what Modules 1.2–1.4's hand-constructed examples deliberately controlled for, in order to isolate one failure mode at a time).

## Part C — Run the full metric suite

This is every metric type from Modules 1.1–1.4, run together against the same golden dataset in one pass — the full picture instead of one metric at a time on hand-picked examples.

In [ ]:
# ============ FULL METRIC SUITE ============
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    FaithfulnessMetric,
    HallucinationMetric,
)
from deepeval.metrics.ragas import RAGASAnswerRelevancyMetric

contextual_precision = ContextualPrecisionMetric(threshold=0.5, include_reason=True, model="gpt-4o")
contextual_recall = ContextualRecallMetric(threshold=0.5, include_reason=True, model="gpt-4o")
contextual_relevancy = ContextualRelevancyMetric(threshold=0.5, include_reason=True, model="gpt-4o")
answer_relevancy = AnswerRelevancyMetric(threshold=0.5, include_reason=True, model="gpt-4o")
ragas_answer_relevancy = RAGASAnswerRelevancyMetric(threshold=0.5, embeddings=openai_embed_model, model="gpt-4o")
faithfulness = FaithfulnessMetric(threshold=0.5, include_reason=True, model="gpt-4o")
hallucination = HallucinationMetric(threshold=0.5, include_reason=True, model="gpt-4o")

eval_results = evaluate(
    test_cases=eval_dataset.test_cases,
    metrics=[
        contextual_precision, contextual_recall, contextual_relevancy,
        answer_relevancy, ragas_answer_relevancy, faithfulness, hallucination,
    ],
)

### Turn the results into a comparable table

In [ ]:
# ============ RESULTS AS A DATAFRAME ============
import pandas as pd

eval_metrics = []
for result in eval_results.test_results:
    row = {
        "Input": result.input,
        "Expected Output": result.expected_output,
        "Actual Output": result.actual_output,
        "Success": result.success,
    }
    for metric in result.metrics_data:
        row[f"{metric.name}_Score"] = metric.score
        row[f"{metric.name}_Success"] = metric.success
    eval_metrics.append(row)

eval_results_df = pd.DataFrame(eval_metrics)
eval_results_df

In [ ]:
# ============ SUMMARY STATISTICS ACROSS THE GOLDEN SET ============
score_columns = [c for c in eval_results_df.columns if c.endswith("_Score")]
eval_results_df[score_columns].describe()

**Reading the results:** this `describe()` table is what an aggregate RAG quality report looks like in practice — mean/min/max per metric, across every golden question at once, instead of eyeballing one test case's score at a time. A low mean on `Contextual Precision_Score` with a high mean everywhere else would point at a ranking problem specifically (Module 1.2); a low mean on `Faithfulness_Score` alongside high `Contextual Relevancy_Score` would point at the generator hallucinating despite good retrieval (Module 1.3) — the same diagnostic logic from every earlier notebook in Module 1, now applied at the dataset level instead of one hand-picked example at a time.

## Summary

- **Synthetic golden datasets** (`Synthesizer.generate_goldens_from_contexts`) solve the "someone has to write reference answers" cost from Module 0 §2 — generate them from your own documents instead of writing them by hand, at the cost of the generated questions being only as good as the synthesis process (worth spot-checking, not blindly trusting).
- Building `retrieval_context` from **live pipeline runs** (not from the documents a golden was generated from) is what makes this an end-to-end evaluation of the real system, not just a generator-quality check under ideal conditions.
- Running the **full metric suite together** (retriever + generator, reference-based + referenceless) and reading it as a table is how you actually diagnose *where* a RAG system is weak, using the same per-metric distinctions taught individually across Modules 1.1–1.4.
- This closes Module 1. Module 2 moves from RAG-specific evaluation to conversational, tool-use, and task-completion evaluation for agents.